# Agentic Retrieval Baseline for Omnilex Legal Retrieval

This notebook implements an **agentic retrieval approach** using a ReAct-style agent with search tools.

## Approach
1. Load a local LLM (GGUF format via llama-cpp-python)
2. Build BM25 search indices for laws and court decisions
3. Create search tools the agent can use
4. For each query, run a ReAct agent that:
   - Reasons about what to search
   - Uses tools to search laws and court decisions
   - Extracts citations from search results
   - Provides final answer with all found citations

## Advantages over Direct Generation
- Grounded in actual legal documents
- Less hallucination of non-existent citations
- Can iterate on searches to find more relevant sources

## Requirements
- llama-cpp-python
- rank-bm25
- A GGUF model file (e.g., Mistral-7B-Instruct)

## 1. Setup & Configuration

In [37]:
import os
import sys
import re
from pathlib import Path

# === CONFIGURATION ===
# Choose which dataset to run on: "val" or "test"
DATASET_MODE = "val"  # Change to "test" for final submission

# Set to True to rebuild indices from CSV (required on first run)
# Set to False to load cached indices (faster for subsequent runs)
FORCE_REBUILD_INDICES = False

# Detect environment
KAGGLE_ENV = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if KAGGLE_ENV:
    # Kaggle paths
    DATA_PATH = Path("/kaggle/input/omnilex-data")
    MODEL_PATH = Path("/kaggle/input/llama-model")
    OUTPUT_PATH = Path("/kaggle/working")
    INDEX_PATH = Path("/kaggle/input/omnilex-indices")
    sys.path.insert(0, "/kaggle/input/omnilex-utils")
else:
    # Local development paths
    REPO_ROOT = Path(".").resolve().parent
    DATA_PATH = REPO_ROOT / "data"
    MODEL_PATH = REPO_ROOT / "models"
    OUTPUT_PATH = REPO_ROOT / "output"
    INDEX_PATH = REPO_ROOT / "data" / "processed"

# CSV corpus files for index building
LAWS_CSV = DATA_PATH / "laws_de.csv"
COURTS_CSV = DATA_PATH / "court_considerations.csv"

# Index cache paths
LAWS_INDEX_PATH = INDEX_PATH / "laws_index.pkl"
COURTS_INDEX_PATH = INDEX_PATH / "courts_index.pkl"

# Derived paths based on DATASET_MODE
QUERY_FILE = DATA_PATH / f"{DATASET_MODE}.csv"
IS_VALIDATION_MODE = DATASET_MODE == "val"

# Create output directory
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
INDEX_PATH.mkdir(parents=True, exist_ok=True)

print(f"Environment: {'Kaggle' if KAGGLE_ENV else 'Local'}")
print(f"Dataset mode: {DATASET_MODE}")
print(f"Query file: {QUERY_FILE}")
print(f"Validation mode: {IS_VALIDATION_MODE}")
print(f"Force rebuild indices: {FORCE_REBUILD_INDICES}")
print(f"\nCorpus files:")
print(f"  Laws CSV: {LAWS_CSV} ({LAWS_CSV.stat().st_size / 1e6:.1f} MB)" if LAWS_CSV.exists() else f"  Laws CSV: {LAWS_CSV} (NOT FOUND)")
print(f"  Courts CSV: {COURTS_CSV} ({COURTS_CSV.stat().st_size / 1e9:.2f} GB)" if COURTS_CSV.exists() else f"  Courts CSV: {COURTS_CSV} (NOT FOUND)")
print(f"\nIndex cache: {INDEX_PATH}")

Environment: Local
Dataset mode: val
Query file: C:\Users\david\Desktop\project_ssl\Omnilex-Agentic-Retrieval-Competition\data\val.csv
Validation mode: True
Force rebuild indices: False

Corpus files:
  Laws CSV: C:\Users\david\Desktop\project_ssl\Omnilex-Agentic-Retrieval-Competition\data\laws_de.csv (73.0 MB)
  Courts CSV: C:\Users\david\Desktop\project_ssl\Omnilex-Agentic-Retrieval-Competition\data\court_considerations.csv (2.43 GB)

Index cache: C:\Users\david\Desktop\project_ssl\Omnilex-Agentic-Retrieval-Competition\data\processed


In [3]:
# Configuration
CONFIG = {
    # Model settings
    "model_file": "mistral-7b-instruct-v0.2.Q4_K_M.gguf",
    "n_ctx": 8192,         # Context window size
    "n_threads": 4,
    "n_gpu_layers": -1,    # GPU layers (-1 = offload all layers to GPU)
    
    # Agent settings
    "max_iterations": 8,   # Max agent iterations per query
    "max_tokens": 512,
    "temperature": 0.1,
    "max_observation_chars": 1200,  # Reduced from 2000 to prevent context overflow
    "max_conversation_chars": 28000,  # Safety net: truncate if conversation exceeds this
    
    # Retrieval settings
    "top_k_laws": 15,       # Results per law search
    "top_k_courts": 15,     # Results per court search
    "top_k_hyde": 20,           # Dense retrieval candidate pool
    "nprobes": 20,              # IVF clusters to search
    "hyde_max_tokens": 120,     # Hypothetical paragraph tokens
    
    # Paths
    "test_file": "test.csv",

    # SAC (Summary-Augmented Chunking)
    "use_sac": True,
    "use_query_decomposition": True,
    "top_k_subquery_bm25": 15,
    "top_k_dense_per_subquery": 10,
    "synthesis_max_tokens": 300

}

## 2. Load Corpora and Build/Load Indices

In [4]:
import pandas as pd
from tqdm.notebook import tqdm
import pickle
import re

# bm25s uses scipy sparse matrices — 10-100x faster than rank_bm25 for both
# index building and query scoring. Install with: pip install bm25s
try:
    import bm25s
    _USE_BM25S = True
except ImportError:
    from rank_bm25 import BM25Okapi
    _USE_BM25S = False
    print("WARNING: bm25s not found, falling back to rank_bm25 (slow). "
          "Install with: pip install bm25s")

print(f"BM25 backend: {'bm25s (fast sparse)' if _USE_BM25S else 'rank_bm25 (slow)'}")


class BM25Index:
    """BM25 index for keyword search over legal documents.

    Uses bm25s (sparse scipy matrices) when available for 10-100x speedup
    over rank_bm25 on both index build and query time.
    """

    def __init__(
        self,
        documents: list[dict] | None = None,
        text_field: str = "text",
        citation_field: str = "citation",
    ):
        self.text_field = text_field
        self.citation_field = citation_field
        self.documents: list[dict] = []
        self._retriever = None  # bm25s.BM25 or BM25Okapi

        if documents:
            self.build(documents)

    def _tokenize(self, text: str) -> list[str]:
        text = text.lower()
        tokens = re.split(r"\W+", text)
        return [t for t in tokens if t]

    def build(self, documents: list[dict]) -> None:
        self.documents = documents
        texts = [doc.get(self.text_field, "") for doc in documents]

        if _USE_BM25S:
            # bm25s tokenizes and builds the sparse inverted index in one pass
            tokenized = bm25s.tokenize(texts, stopwords=None, lower=True, show_progress=False)
            self._retriever = bm25s.BM25()
            self._retriever.index(tokenized)
        else:
            tokenized_corpus = [self._tokenize(t) for t in texts]
            self._retriever = BM25Okapi(tokenized_corpus)
            self._tokenized_corpus = tokenized_corpus

    def search(
        self,
        query: str,
        top_k: int = 10,
        return_scores: bool = False,
    ) -> list[dict]:
        if self._retriever is None:
            raise ValueError("Index not built. Call build() first.")
        if not query or not query.strip():
            return []

        if _USE_BM25S:
            query_tokens = bm25s.tokenize([query], stopwords=None, lower=True, show_progress=False)
            n = min(top_k, len(self.documents))
            try:
                doc_indices, scores = self._retriever.retrieve(query_tokens, k=n)
            except Exception:
                return []

            results = []
            for idx, score in zip(doc_indices[0], scores[0]):
                if score <= 0:
                    continue
                doc = self.documents[int(idx)].copy()
                if return_scores:
                    doc["_score"] = float(score)
                results.append(doc)
            return results
        else:
            query_tokens = self._tokenize(query)
            if not query_tokens:
                return []
            scores = self._retriever.get_scores(query_tokens)
            top_indices = scores.argsort()[-top_k:][::-1]
            results = []
            for idx in top_indices:
                if scores[idx] <= 0:
                    continue
                doc = self.documents[idx].copy()
                if return_scores:
                    doc["_score"] = float(scores[idx])
                results.append(doc)
            return results

    def save(self, path) -> None:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)

        data = {
            "documents": self.documents,
            "text_field": self.text_field,
            "citation_field": self.citation_field,
            "backend": "bm25s" if _USE_BM25S else "rank_bm25",
        }

        if _USE_BM25S:
            # Save retriever to a companion directory, documents to pickle
            retriever_dir = str(path) + "_bm25s"
            self._retriever.save(retriever_dir, corpus=None)
            data["retriever_dir"] = retriever_dir
        else:
            data["tokenized_corpus"] = self._tokenized_corpus

        with open(path, "wb") as f:
            pickle.dump(data, f)

    @classmethod
    def load(cls, path) -> "BM25Index":
        path = Path(path)

        with open(path, "rb") as f:
            data = pickle.load(f)

        instance = cls(
            text_field=data["text_field"],
            citation_field=data.get("citation_field", "citation"),
        )
        instance.documents = data["documents"]

        if _USE_BM25S and data.get("backend") == "bm25s" and "retriever_dir" in data:
            instance._retriever = bm25s.BM25.load(data["retriever_dir"], load_corpus=False)
        elif _USE_BM25S:
            # Old rank_bm25 pickle — rebuild index with bm25s (one-time cost)
            print("  Old rank_bm25 pickle detected — rebuilding with bm25s (one-time)...")
            texts = [doc.get(instance.text_field, "") for doc in instance.documents]
            tokenized = bm25s.tokenize(texts, stopwords=None, lower=True, show_progress=True)
            instance._retriever = bm25s.BM25()
            instance._retriever.index(tokenized)
        else:
            tokenized_corpus = data["tokenized_corpus"]
            instance._retriever = BM25Okapi(tokenized_corpus)
            instance._tokenized_corpus = tokenized_corpus

        return instance


def load_csv_corpus(
    csv_path: Path,
    chunk_size: int = 100_000,
    max_rows: int | None = None
) -> list[dict]:
    """Load CSV corpus into list of dicts with progress bar.

    Uses column-vectorized iteration (zip) instead of iterrows — ~20x faster.
    """
    documents = []

    print(f"Counting rows in {csv_path.name}...")
    with open(csv_path, encoding="utf-8") as f:
        total_rows = sum(1 for _ in f) - 1  # minus header

    if max_rows:
        total_rows = min(total_rows, max_rows)
    print(f"Total rows to load: {total_rows:,}")

    rows_loaded = 0
    with tqdm(total=total_rows, desc=f"Loading {csv_path.name}") as pbar:
        for chunk in pd.read_csv(csv_path, chunksize=chunk_size, dtype=str):
            remaining = total_rows - rows_loaded
            chunk = chunk.iloc[: min(len(chunk), remaining)]

            # Vectorized fill for NaN text values
            citations = chunk["citation"].tolist()
            texts = chunk["text"].fillna("").tolist()

            for citation, text in zip(citations, texts):
                documents.append({"citation": citation, "text": text})

            rows_loaded += len(chunk)
            pbar.update(len(chunk))
            if max_rows and rows_loaded >= max_rows:
                break

    return documents


def build_sac_documents(documents: list[dict]) -> list[dict]:
    """Summary-Augmented Chunking: prepend a 150-char decision summary to every chunk.

    Groups rows sharing the same base decision (same docket/BGE prefix before
    ' E.') into a group, builds a 150-char summary from concatenated group text,
    and prepends [KONTEXT: <summary>] to each chunk's text field.
    Only intended for the courts corpus.
    """
    import re as _re_sac
    from collections import defaultdict

    def _base_citation(cit: str) -> str:
        m = _re_sac.match(r'^(.+?)\s+E\.\s+', cit)
        return m.group(1) if m else cit

    groups: dict[str, list[int]] = defaultdict(list)
    for idx, doc in enumerate(documents):
        groups[_base_citation(doc.get('citation', ''))].append(idx)

    n_groups = len(groups)
    result = [doc.copy() for doc in documents]
    for base, indices in groups.items():
        full_text = ' '.join(documents[i].get('text', '') for i in indices)
        summary = ' '.join(full_text.split())[:150]
        prefix = '[KONTEXT: ' + summary + '] '
        for i in indices:
            result[i]['text'] = prefix + result[i]['text']

    print(f'  SAC: {n_groups:,} unique decision groups across {len(documents):,} chunks')
    return result


def get_or_build_index(
    name: str,
    csv_path: Path,
    index_path: Path,
    force_rebuild: bool = False,
    max_rows: int | None = None,
    preprocessor=None,
) -> BM25Index:
    """Load cached index or build from CSV."""
    if index_path.exists() and not force_rebuild:
        print(f"Loading cached {name} index from {index_path}")
        try:
            index = BM25Index.load(index_path)
            print(f"  Loaded {len(index.documents):,} documents")
            return index
        except MemoryError:
            print(f"  WARNING: not enough RAM to load cached {name} index — "
                  f"falling back to empty index (HyDE/lookup still available).")
            return BM25Index(documents=[])

    if not csv_path.exists():
        print(f"Warning: {csv_path} not found. Creating empty index.")
        return BM25Index(documents=[])

    print(f"\n{'='*50}")
    print(f"Building {name} index from {csv_path}")
    print(f"{'='*50}")
    documents = load_csv_corpus(csv_path, max_rows=max_rows)

    if not documents:
        print(f"Warning: No documents loaded. Creating empty index.")
        return BM25Index(documents=[])

    if preprocessor is not None:
        print(f'  Applying {preprocessor.__name__} preprocessing...')
        documents = preprocessor(documents)

    print(f"\nBuilding BM25 index for {len(documents):,} documents...")
    index = BM25Index(documents=documents, text_field="text", citation_field="citation")
    print(f"Index built successfully!")

    if not KAGGLE_ENV:
        print(f"Saving index to {index_path}...")
        index.save(index_path)
        print(f"Index cached.")

    return index


resource module not available on Windows
BM25 backend: bm25s (fast sparse)


In [5]:
# Load or build laws index
# Laws CSV: ~45MB, ~269K rows
# Build time: ~30 seconds | Load from cache: <1 second

laws_index = get_or_build_index(
    name="laws",
    csv_path=LAWS_CSV,
    index_path=LAWS_INDEX_PATH,
    force_rebuild=FORCE_REBUILD_INDICES,
    # max_rows=10000  # Uncomment to test with smaller corpus
)
print(f"\nLaws index: {len(laws_index.documents):,} documents")

# Test search
test_results = laws_index.search("Vertrag", top_k=3)
print(f"\nTest search 'Vertrag': {len(test_results)} results")
if test_results:
    print(f"  Top result: {test_results[0].get('citation', 'N/A')}")

Loading cached laws index from C:\Users\david\Desktop\project_ssl\Omnilex-Agentic-Retrieval-Competition\data\processed\laws_index.pkl
  Loaded 175,933 documents

Laws index: 175,933 documents


BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


Test search 'Vertrag': 3 results
  Top result: Art. 17 Abs. 6 VID


In [6]:
# Load or build courts index
# Courts CSV: ~2.3GB, ~2.5M rows
# With bm25s — Full corpus build time: ~3-5 min | Peak RAM: ~3-4GB | Load from cache: <5 seconds
# With rank_bm25 (fallback) — Full corpus build time: ~15-20 min | Peak RAM: ~8-16GB

courts_index = get_or_build_index(
    name="courts",
    csv_path=COURTS_CSV,
    index_path=COURTS_INDEX_PATH,
    force_rebuild=FORCE_REBUILD_INDICES,
    preprocessor=build_sac_documents if CONFIG.get("use_sac") else None,
    # max_rows=100000  # Uncomment to test with smaller corpus
)
print(f"\nCourts index: {len(courts_index.documents):,} documents")

# Test search
test_results = courts_index.search("Meinungsfreiheit", top_k=3)
print(f"\nTest search 'Meinungsfreiheit': {len(test_results)} results")
if test_results:
    print(f"  Top result: {test_results[0].get('citation', 'N/A')}")


Loading cached courts index from C:\Users\david\Desktop\project_ssl\Omnilex-Agentic-Retrieval-Competition\data\processed\courts_index.pkl
  Loaded 2,476,315 documents

Courts index: 2,476,315 documents


BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


Test search 'Meinungsfreiheit': 3 results
  Top result: BGE 148 I 33 E. 6.1


In [7]:
# ── Citation lookup dicts — O(1) direct access by citation key ────────────
# RAM cost: laws ~80MB, courts ~800MB — well within 16GB budget
print('Building citation lookup dicts...')

laws_lookup = {
    doc['citation']: doc['text']
    for doc in laws_index.documents
    if doc.get('citation')
}

courts_lookup = {
    doc['citation']: doc['text']
    for doc in courts_index.documents
    if doc.get('citation')
}

print(f'  laws_lookup   : {len(laws_lookup):,} entries')
print(f'  courts_lookup : {len(courts_lookup):,} entries')

# Verify key gold citations are reachable
test_keys = ['Art. 221 Abs. 1 StPO', 'Art. 100 Abs. 1 BGG', 'Art. 37 Abs. 1 StBOG']
for k in test_keys:
    found = k in laws_lookup
    print(f"  {'OK' if found else 'MISSING'} laws_lookup['{k}']")


Building citation lookup dicts...
  laws_lookup   : 175,933 entries
  courts_lookup : 1,985,178 entries
  OK laws_lookup['Art. 221 Abs. 1 StPO']
  OK laws_lookup['Art. 100 Abs. 1 BGG']
  OK laws_lookup['Art. 37 Abs. 1 StBOG']


In [59]:
# ── Dense retrieval components (optional — degrades to BM25-only if unavailable)
import torch

try:
    import lancedb
    from sentence_transformers import SentenceTransformer, CrossEncoder

    if KAGGLE_ENV:
        LANCEDB_PATH = Path("/kaggle/input/omnilex-indices/lancedb_courts")
    else:
        LANCEDB_PATH = REPO_ROOT / "data" / "processed" / "lancedb_courts"

    if LANCEDB_PATH.exists():
        _lance_db    = lancedb.connect(str(LANCEDB_PATH))
        _lance_table = _lance_db.open_table("courts")
        _device      = "cuda" if torch.cuda.is_available() else "cpu"
        _embed_model = SentenceTransformer(
            "intfloat/multilingual-e5-base", device=_device
        )
        _embed_model.max_seq_length = 512
        _DENSE_AVAILABLE = True
        print(f"Dense retrieval ready  (device={_device})")
    else:
        _DENSE_AVAILABLE = False
        _lance_table = _embed_model = None
        print("LanceDB not found — dense retrieval disabled (BM25-only mode)")

    try:
        # Load the multilingual MiniLM cross-encoder
        _cross_encoder = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1", device=_device)
        _CROSS_ENCODER_AVAILABLE = True
        print(f"Cross-encoder reranking ready (device={_device})")
    except Exception as e:
        _CROSS_ENCODER_AVAILABLE = False
        _cross_encoder = None
        print(f"Cross-encoder not available: {e}")
except ImportError:
    _DENSE_AVAILABLE = False
    _lance_table = _embed_model = None
    print("lancedb/sentence_transformers not installed — BM25-only mode")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Dense retrieval ready  (device=cuda)


config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

c:\Users\david\miniconda3\envs\omnilex\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\david\.cache\huggingface\hub\models--cross-encoder--mmarco-mMiniLMv2-L12-H384-v1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


...

tokenizer_config.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Cross-encoder reranking ready (device=cuda)


## 3. Define Search Tools

In [9]:
import re as _re_tools


class CitationLookupTool:
    """O(1) direct lookup by citation key with normalization and prefix matching."""

    name: str = "lookup_citation"
    description: str = (
        "Look up the exact text of a specific legal citation.\n"
        "Input: citation string e.g. \"Art. 221 Abs. 1 StPO\" or \"BGE 137 IV 122 E. 6.2\"\n"
        "Output: text of that provision, or a list of matching keys for prefix searches.\n"
        "Use this when you know which article or decision is likely relevant."
    )
    _fr_to_de = {
        "LAI": "IVG", "CO": "OR", "CC": "ZGB", "CP": "StGB",
        "CPP": "StPO", "LPGA": "ATSG", "LTF": "BGG",
        "LAA": "UVG", "LAVS": "AHVG", "LPP": "BVG",
    }

    def __init__(self, laws_lut, courts_lut, max_prefix_results=3):
        self._laws   = laws_lut
        self._courts = courts_lut
        self.max_prefix_results = max_prefix_results
        self._last_citations: list[str] = []

    def __call__(self, citation: str) -> str:
        return self.run(citation)

    def _normalize(self, citation: str) -> str:
        for stop in ["\n", "Thought:", "Action:", "Now ", "Found ", "Looking", "Next"]:
            if stop in citation:
                citation = citation[:citation.index(stop)]
        citation = citation.strip()
        citation = _re_tools.sub(r"\s*\([^)]*\)", "", citation).strip()
        citation = _re_tools.sub(r"[\s]*[:\.\,;]$", "", citation).strip()
        citation = _re_tools.sub(r"\s+lit\.\s+\w+.*$", "", citation).strip()
        citation = _re_tools.sub(r"\s+Ziff\.\s+\w+.*$", "", citation).strip()
        tokens = citation.split()
        if tokens and tokens[-1] in self._fr_to_de:
            tokens[-1] = self._fr_to_de[tokens[-1]]
            citation = " ".join(tokens)
        return citation

    def run(self, citation: str) -> str:
        citation = self._normalize(citation)
        if not citation:
            return "Error: empty citation string."
        self._last_citations = []

        # 1. Exact match
        for store in [self._laws, self._courts]:
            if citation in store:
                text = store[citation]
                if len(text) > 200:
                    text = text[:200] + "..."
                self._last_citations = [citation]
                return f"[{citation}]\n{text}"

        # 2. Law-code-aware prefix match
        _art = _re_tools.match(r"^(Art\.\s+\d+\w*)\s+([A-Z]\w+(?:bis|ter)?)$", citation)
        candidates = []
        for store in [self._laws, self._courts]:
            if _art:
                pfx, sfx = _art.group(1), _art.group(2)
                candidates += [(k, v) for k, v in store.items()
                               if k.startswith(pfx) and sfx in k]
            else:
                candidates += [(k, v) for k, v in store.items()
                               if k.startswith(citation)]

        if candidates:
            self._last_citations = [k for k, _ in candidates[:self.max_prefix_results]]
            key_list = "\n".join(
                f"  CITATION_KEY: {k}" for k, _ in candidates[:self.max_prefix_results]
            )
            return (
                f"Prefix \"{citation}\" matched {len(candidates)} citations. "
                f"Top {min(len(candidates), self.max_prefix_results)} keys:\n{key_list}\n\n"
                f"Call lookup_citation with the EXACT key, e.g. lookup_citation(\"Art. 100 Abs. 1 BGG\")"
            )

        return (
            f"Citation \"{citation}\" not found. "
            f"Valid formats: \"Art. 221 Abs. 1 StPO\", \"BGE 137 IV 122 E. 6.2\", \"1B_90/2021 E. 2.1\""
        )

    def get_last_citations(self) -> list[str]:
        return list(self._last_citations)


class LawSearchTool:
    """BM25 search over Swiss federal laws."""

    name: str = "search_laws"
    description: str = (
        "Search Swiss federal laws by German keywords.\n"
        "Input: German legal keywords  Output: law citations with excerpts"
    )

    def __init__(self, index, top_k=15, max_excerpt=300):
        self.index = index
        self.top_k = top_k
        self.max_excerpt = max_excerpt
        self._last_results: list[dict] = []

    def __call__(self, query: str) -> str:
        return self.run(query)

    def run(self, query: str) -> str:
        if not query or not query.strip():
            return "Error: empty query."
        self._last_results = self.index.search(query, top_k=self.top_k)
        if not self._last_results:
            return f"No laws found for: {query!r}"
        parts = []
        for doc in self._last_results:
            text = doc.get("text", "")[:self.max_excerpt]
            parts.append(f"- {doc.get('citation','?')}: {text}")
        return "\n".join(parts)

    def get_last_citations(self) -> list[str]:
        return [d.get("citation", "") for d in self._last_results if d.get("citation")]


class CourtSearchTool:
    """BM25 search over Swiss court decisions."""

    name: str = "search_courts"
    description: str = (
        "Search Swiss Federal Court decisions by German keywords.\n"
        "Input: German legal keywords  Output: court citations with excerpts"
    )

    def __init__(self, index, top_k=15, max_excerpt=300):
        self.index = index
        self.top_k = top_k
        self.max_excerpt = max_excerpt
        self._last_results: list[dict] = []

    def __call__(self, query: str) -> str:
        return self.run(query)

    def run(self, query: str) -> str:
        if not query or not query.strip():
            return "Error: empty query."
        self._last_results = self.index.search(query, top_k=self.top_k)
        if not self._last_results:
            return f"No court decisions found for: {query!r}"
        parts = []
        for doc in self._last_results:
            text = doc.get("text", "")[:self.max_excerpt]
            parts.append(f"- {doc.get('citation','?')}: {text}")
        return "\n".join(parts)

    def get_last_citations(self) -> list[str]:
        return [d.get("citation", "") for d in self._last_results if d.get("citation")]


class HyDESearchTool:
    """Dense court search via Hypothetical Document Embeddings + BM25 RRF fusion."""

    name: str = "hyde_search_courts"
    description: str = (
        "Search court decisions by meaning using semantic similarity.\n"
        "Input: 5-8 German legal keywords (NOT citation strings)\n"
        "Output: most relevant court decision sections\n"
        "Use when BM25 search fails or for landmark BGE decisions.\n"
        "Example: \"Kollusionsgefahr Untersuchungshaft Verhaeltnismaessigkeit\""
    )

    def __init__(self, llm, embed_model, lance_table, top_k=20, nprobes=20,
                 hyde_tokens=120, max_excerpt=350, dense_available=True):
        self._llm           = llm
        self._embed         = embed_model
        self._table         = lance_table
        self.top_k          = top_k
        self.nprobes        = nprobes
        self.hyde_tokens    = hyde_tokens
        self.max_excerpt    = max_excerpt
        self.dense_available = dense_available
        self._last_citations: list[str] = []

    def __call__(self, query: str) -> str:
        return self.run(query)

    def _generate_hypothetical(self, query: str) -> str:
        prompt = (
            "[INST] Du bist ein Schweizer Bundesrichter. "
            "Schreibe einen Erwaegungsabsatz (ca. 80 Woerter) auf Deutsch "
            "der die folgende Rechtsfrage beantwortet. "
            "Nur der Erwaegungstext, keine Einleitung.\n\n"
            f"Rechtsfrage: {query} [/INST]"
        )
        out = self._llm(prompt, max_tokens=self.hyde_tokens,
                        temperature=0.3, echo=False)
        return out["choices"][0]["text"].strip()

    def run(self, query: str) -> str:
        query = query.strip().splitlines()[0].strip()
        if not query:
            return "Error: empty query."
        self._last_citations = []

        if self.dense_available and self._embed and self._table:
            hyp = self._generate_hypothetical(query)
            vec = self._embed.encode(
                ["query: " + hyp], normalize_embeddings=True,
                show_progress_bar=False
            )[0].tolist()
            df = (self._table.search(vec).nprobes(self.nprobes)
                  .limit(self.top_k).to_pandas())
            dense = df.to_dict("records") if not df.empty else []
        else:
            hyp   = query
            dense = []

        # BM25 fallback / fusion
        bm25 = courts_index.search(query, top_k=self.top_k)
        fused = reciprocal_rank_fusion(bm25, dense, rrf_k=60)[:self.top_k]

        # Distance filter (dense-only columns)
        DIST_THRESHOLD = 0.35
        top = []
        for doc in fused:
            d = doc.get("_distance", 0.0)
            if "_distance" not in doc or d < DIST_THRESHOLD:
                top.append(doc)
        top = top[:5] or fused[:3]

        self._last_citations = [c.get("citation", "") for c in top]
        parts = []
        for c in top:
            text = c.get("text", "")[:self.max_excerpt]
            parts.append(f"[CITATION_KEY: {c.get('citation','?')}]\n{text}")

        header = f"Found {len(top)} relevant court sections"
        if self.dense_available:
            header += f". Hypothetical: \"{hyp[:60]}...\"\n"
        return header + "\n\n" + "\n\n".join(parts)

    def get_last_citations(self) -> list[str]:
        return list(self._last_citations)


# Instantiate BM25 tools (LLM-independent)
law_tool   = LawSearchTool(laws_index,   top_k=CONFIG["top_k_laws"])
court_tool = CourtSearchTool(courts_index, top_k=CONFIG["top_k_courts"])
lookup_tool = CitationLookupTool(
    laws_lut=laws_lookup, courts_lut=courts_lookup, max_prefix_results=3
)
print("BM25 tools ready. HyDE tool will be instantiated after LLM loads.")


BM25 tools ready. HyDE tool will be instantiated after LLM loads.


## 4. Load Local LLM

In [10]:
from llama_cpp import Llama
import importlib.util


def has_cuda_support() -> bool:
    """Check if llama-cpp-python was built with CUDA support.

    Returns:
        True if CUDA support is available, False otherwise
    """
    try:
        spec = importlib.util.find_spec("llama_cpp")
        if spec and spec.origin:
            lib_dir = Path(spec.origin).parent
            # Check for CUDA shared libraries in main dir and lib/ subdirectory
            cuda_libs = (
                list(lib_dir.glob("*cuda*"))
                + list(lib_dir.glob("*cublas*"))
                + list((lib_dir / "lib").glob("*cuda*"))
                + list((lib_dir / "lib").glob("*cublas*"))
            )
            if cuda_libs:
                return True
        return False
    except Exception:
        return False


def get_device_info(n_gpu_layers: int) -> str:
    """Get human-readable device info string.

    Args:
        n_gpu_layers: Number of GPU layers configured

    Returns:
        String describing the compute device
    """
    if n_gpu_layers == -1:
        return "GPU (all layers offloaded)"
    elif n_gpu_layers > 0:
        return f"GPU ({n_gpu_layers} layers offloaded)"
    else:
        return "CPU"

# Find model file
model_file = MODEL_PATH / CONFIG["model_file"]

if not model_file.exists():
    gguf_files = list(MODEL_PATH.glob("*.gguf")) + list(MODEL_PATH.rglob("*.gguf"))
    if gguf_files:
        model_file = gguf_files[0]
        print(f"Using model: {model_file}")
    else:
        raise FileNotFoundError(
            f"No model found. Please download a GGUF model to {MODEL_PATH}"
        )

print(f"Loading model: {model_file}")

# Auto-detect GPU: use GPU if available, else CPU
n_gpu_layers = CONFIG["n_gpu_layers"]
if n_gpu_layers == -1 and not has_cuda_support():
    n_gpu_layers = 0  # Fallback to CPU if no CUDA support

llm = Llama(
    model_path=str(model_file),
    n_ctx=CONFIG["n_ctx"],
    n_threads=CONFIG["n_threads"],
    n_gpu_layers=n_gpu_layers,
    verbose=False,
)

print("Model loaded successfully!")
print(f"Running on: {get_device_info(n_gpu_layers)}")

Loading model: C:\Users\david\Desktop\project_ssl\Omnilex-Agentic-Retrieval-Competition\models\mistral-7b-instruct-v0.2.Q4_K_M.gguf


llama_new_context_with_model: n_ctx_per_seq (8192) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Model loaded successfully!
Running on: GPU (all layers offloaded)


In [11]:
# Instantiate HyDE tool now that LLM is loaded
hyde_tool = HyDESearchTool(
    llm            = llm,
    embed_model    = _embed_model,
    lance_table    = _lance_table,
    top_k          = CONFIG.get("top_k_hyde", 20),
    nprobes        = CONFIG.get("nprobes", 20),
    hyde_tokens    = CONFIG.get("hyde_max_tokens", 120),
    max_excerpt    = CONFIG.get("max_observation_chars", 1200),
    dense_available = _DENSE_AVAILABLE,
)

TOOLS = {
    "search_laws":        law_tool,
    "search_courts":      court_tool,
    "lookup_citation":    lookup_tool,
    "hyde_search_courts": hyde_tool,
}
print("All tools ready:", list(TOOLS.keys()))


All tools ready: ['search_laws', 'search_courts', 'lookup_citation', 'hyde_search_courts']


In [12]:
import re
def translate_query_to_keywords(query: str, llm, verbose: bool = False) -> str:
    """
    Translates an English legal query into 5-8 highly relevant German keywords
    using a Few-Shot prompt to prevent hallucinated article numbers.
    """
    prompt = (
        "[INST] You are a Swiss legal search assistant. Extract 5-8 German keywords "
        "and the overarching law abbreviation (e.g., StPO, ZGB, OR, IVG) from the English query. "
        "Output ONLY a space-separated list of words. NEVER invent specific article numbers.\n\n"
        "Example:\n"
        "Query: Can a landlord evict me if I am two weeks late on rent?\n"
        "Keywords: Mietvertrag Kündigung Zahlungsverzug Ausweisung Miete OR\n\n"
        f"Query: {query}\n"
        "Keywords: [/INST]"
    )

    try:
        response = llm(
            prompt,
            max_tokens=25,       
            temperature=0.0,     # 0.0 forces the most predictable path
            stop=["\n", "</s>", "Query:"], 
            echo=False
        )
        
        raw_output = response["choices"][0]["text"].strip()
        cleaned_keywords = re.sub(r'[^a-zA-Z0-9äöüÄÖÜß\s\-]', ' ', raw_output)
        cleaned_keywords = ' '.join(cleaned_keywords.split())

        if verbose:
            print(f"Query   : {query}")
            print(f"Keywords: {cleaned_keywords}")

        return cleaned_keywords

    except Exception as e:
        print(f"Error: {e}")
        return query

In [13]:
import json as _json


def decompose_query_to_subqueries(query: str, llm) -> list[str]:
    """
    Decomposes an English legal query into 3-4 distinct German sub-queries,
    each targeting a different legal element of the case.
    Returns a list of German search strings.
    Falls back to [translate_query_to_keywords(query, llm)] on parse failure.
    """
    prompt = (
        "[INST] You are a Swiss legal search assistant. Given an English legal query, "
        "extract 3-4 distinct German search sub-queries. Each sub-query should target "
        "a different legal element or issue in the case. "
        "Output ONLY a JSON array of strings, no explanation.\n\n"
        "Example:\n"
        "Query: \"Can a court lawfully order a three-month extension of pre-trial "
        "detention for risk of collusion?\"\n"
        "Output: [\"Untersuchungshaft Verlängerung Kollusionsgefahr\", "
        "\"Verhältnismäßigkeit Untersuchungshaft StPO\", "
        "\"Haftgrund Fluchtgefahr Verdunkelungsgefahr\", "
        "\"Haftprüfung Bundesgericht BGG\"]\n\n"
        f"Query: {query}\n"
        "Output: [/INST]"
    )

    try:
        response = llm(
            prompt,
            max_tokens=150,
            temperature=0.1,
            stop=["</s>", "[INST]"],
            echo=False,
        )
        raw = response["choices"][0]["text"].strip()
        start = raw.find("[")
        end   = raw.rfind("]") + 1
        if start != -1 and end > start:
            subqueries = _json.loads(raw[start:end])
            if isinstance(subqueries, list) and subqueries:
                return [str(s).strip() for s in subqueries if str(s).strip()]
    except Exception:
        pass
    return [translate_query_to_keywords(query, llm)]


In [23]:
# Test decompose_query_to_subqueries
print("Testing Query Decomposer:\n" + "="*50)
for q in test_queries:
    print(f"Query: {q}")
    subs = decompose_query_to_subqueries(q, llm)
    for i, s in enumerate(subs, 1):
        print(f"  {i}. {s}")
    print("-" * 50)


Testing Query Decomposer:


NameError: name 'test_queries' is not defined

In [14]:
def reciprocal_rank_fusion(bm25_results: list, dense_results: list, rrf_k: int = 60) -> list:
    """
    Merges BM25 and Dense retrieval results using Reciprocal Rank Fusion.
    RRF Score = 1 / (k + rank)
    """
    fused_scores = {}
    doc_store = {} # To keep track of the actual document text/metadata
    
    # 1. Process BM25 Results
    for rank, doc in enumerate(bm25_results, start=1):
        cit = doc.get("citation")
        if not cit: continue
        
        doc_store[cit] = doc
        fused_scores[cit] = fused_scores.get(cit, 0.0) + (1.0 / (rrf_k + rank))
        
    # 2. Process Dense (LanceDB) Results
    for rank, doc in enumerate(dense_results, start=1):
        cit = doc.get("citation")
        if not cit: continue
        
        if cit not in doc_store:
            doc_store[cit] = doc
            
        fused_scores[cit] = fused_scores.get(cit, 0.0) + (1.0 / (rrf_k + rank))
        
    # 3. Sort by fused score descending
    reranked_citations = sorted(fused_scores.keys(), key=lambda x: fused_scores[x], reverse=True)
    
    # Return the documents in their new fused order
    return [doc_store[cit] for cit in reranked_citations]

def hybrid_search(german_keywords: str, embed_model, lance_table, top_k: int = 50) -> list:
    """
    Executes BM25 and Dense search in parallel, then fuses them.
    """
    if not german_keywords or not german_keywords.strip():
        return []
        
    # --- 1. BM25 Search (Exact Keyword Match) ---
    # We search both laws and courts. 
    # (Assuming laws_index and courts_index are in the global scope from earlier cells)
    bm25_laws = laws_index.search(german_keywords, top_k=top_k)
    bm25_courts = courts_index.search(german_keywords, top_k=top_k)
    bm25_combined = bm25_laws + bm25_courts
    
    # --- 2. Dense Search (Semantic Match) ---
    # Encode the keywords into a vector using e5-base
    # The e5-base model requires the "query: " prefix for queries
    query_vector = embed_model.encode(
        [f"query: {german_keywords}"], 
        normalize_embeddings=True, 
        show_progress_bar=False
    )[0].tolist()
    
    # Search LanceDB courts table
    dense_df = (
        lance_table.search(query_vector)
        .nprobes(20)
        .limit(top_k)
        .to_pandas()
    )
    dense_courts = dense_df.to_dict("records") if not dense_df.empty else []
    
    # --- 3. Fuse Results ---
    # Merge the BM25 and Dense results using RRF
    fused_results = reciprocal_rank_fusion(bm25_combined, dense_courts, rrf_k=60)
    
    # Return the top N after fusion
    return fused_results[:top_k]

In [15]:
# Test the query translator to verify the LLM prompt works
test_queries = [
    "What are the requirements for a valid contract under Swiss law?",
    "Can a court lawfully order a three-month extension of pre-trial detention for risk of collusion?",
    "Does a claimant with allergic asthma have an entitlement to invalidity insurance benefits?"
]

print("Testing Query Translator:\n" + "="*50)
for q in test_queries:
    translate_query_to_keywords(q, llm, verbose=True)
    print("-" * 50)

Testing Query Translator:
Query   : What are the requirements for a valid contract under Swiss law?
Keywords: Vertrag Gebotliche Bedingungen Rechtsgültigkeit Obligationenabkommen Zivilgesetzbuch
--------------------------------------------------
Query   : Can a court lawfully order a three-month extension of pre-trial detention for risk of collusion?
Keywords: Strafverfahren Gerichtsverfassung Richterliches Prüfungsverfahren Bewe
--------------------------------------------------
Query   : Does a claimant with allergic asthma have an entitlement to invalidity insurance benefits?
Keywords: Allergie Asthma Invalidenversicherung Behinderung Sozialversicherung OR
--------------------------------------------------


## 5. Define DAG Pipeline

In [117]:
import re
from collections import Counter
def _filter_valid_citations(citations: list[str]) -> list[str]:
    """Keep only strings that look like real Swiss legal citations."""
    import re as _re
    _VALID_LAWS = {
        # Swiss Federal Constitution
        "BV", "Cst", "Cost",
        # German law codes (Swiss Federal)
        "BGG", "ZGB", "OR", "StGB", "StPO", "IVG", "ATSG", "UVG", "BVG", "KVG",
        "AHVG", "ELG", "DSG", "BPG", "VwVG", "ZPO", "DBG", "StHG", "SchKG",
        "ArG", "RPG", "USG", "StBOG", "BZP", "OHG", "GwG", "BankG", "KAG",
        "MWStG", "UWG", "URG", "PatG", "MSchG", "HMG", "BetmG", "EBG", "FZG",
        "MVG", "EOG", "SVG", "BSG", "SHG", "KG", "ParlG", "BGerR", "IPRG",
        # French abbreviations (Swiss Federal)
        "LAI", "LTF", "CPP", "LAA", "LPP", "LPGA", "LCD", "LDA", "LBI", "LPM",
        "LCA", "LFPr", "LAVS", "LACI", "LEI", "FDPA", "CO", "CC", "CP",
        # Italian abbreviations (Swiss Federal)
        "LI", "LIFD", "CO", "CC", "CP",
        # International conventions
        "EMRK", "AEUV",
        # Other Standard Swiss Entities/Concepts
        "AHV", "EO", "SUVA",
    }
    
    _BAD = ["[", "]", "lookup_citation", "Note:", "CITATION_KEY",
            "\n", "relevant", "obtained", "assuming", "actual"]
    
    seen, out = set(), []
    for cit in citations:
        cit = cit.strip().lstrip("- \u2022*")
        if not cit or cit in seen:
            continue
        if any(b.lower() in cit.lower() for b in _BAD):
            continue
            
        # 1. Standard Law Abbreviations (e.g., Art. X Abs. Y OR)
        art_m = _re.match(
            r"^Art\.\s+\d+[a-z]?\s+(?:(?:Abs\.|Ziff\.)\s+\d+\w*(?:\s+lit\.\s+\w+)?\s+)?([A-Z]\w+)", cit
        )
        if art_m:
            if art_m.group(1) not in _VALID_LAWS:
                continue
            seen.add(cit)
            out.append(cit)
            continue
            
        # 2. SR Numbers (Systematische Rechtssammlung - e.g., SR 210)
        # This is the official Swiss classified compilation numbering system
        if _re.match(r"^SR\s+\d+(\.\d+)*", cit):
            seen.add(cit)
            out.append(cit)
            continue
            
        # 3. BGE Decisions (Swiss Federal Supreme Court decisions)
        if _re.match(r"^BGE\s+\d{2,3}\s+[IVX]+\w*\s+\d+", cit):
            seen.add(cit)
            out.append(cit)
            continue
            
        # 4. Modern Swiss docket format (e.g., 1B_90/2021)
        if _re.match(r"^\d+[A-Z]{1,2}_\d+/\d{4}", cit):
            seen.add(cit)
            out.append(cit)
            continue
            
        # 5. Old Swiss docket format (e.g., 4P.172/2006)
        if _re.match(r"^\d+[A-Z]\.\d+/\d{4}", cit):
            seen.add(cit)
            out.append(cit)
            continue
            
    return out


DOMAIN_STATUTE_MAP = {
    "detention": [
        "Art. 221 Abs. 1 StPO", "Art. 221 Abs. 2 StPO", "Art. 212 Abs. 3 StPO",
        "Art. 226 Abs. 1 StPO", "Art. 227 Abs. 1 StPO", "Art. 228 Abs. 1 StPO",
        "Art. 231 Abs. 1 StPO", "Art. 237 Abs. 1 StPO", "Art. 100 Abs. 1 BGG",
    ],
    "disability": [
        "Art. 17 Abs. 1 IVG", "Art. 8 Abs. 1 IVG", "Art. 4 Abs. 1 IVG",
        "Art. 16 ATSG", "Art. 7 Abs. 1 ATSG", "Art. 6 ATSG", "Art. 28 Abs. 1 IVG",
    ],
    "contract": [
        "Art. 1 Abs. 1 OR", "Art. 18 Abs. 1 OR", "Art. 97 Abs. 1 OR", "Art. 41 Abs. 1 OR",
    ],
    "criminal": [
        "Art. 10 Abs. 2 StGB", "Art. 47 Abs. 1 StGB", "Art. 49 Abs. 1 StGB",
    ],
    # --- NEW DOMAIN ADDED HERE ---
    "inheritance": [
        "Art. 467 ZGB", "Art. 469 Abs. 1 ZGB", "Art. 469 Abs. 2 ZGB", 
        "Art. 471 ZGB", "Art. 505 Abs. 1 ZGB", "Art. 520a ZGB",
    ],
}

DOMAIN_KEYWORDS = {
    "detention": ["detention", "pre-trial", "remand", "arrest", "custody",
                  "collusion", "flight risk"],
    "disability": ["disability", "invalidity", "incapacity", "IV", "ATSG",
                   "insurance benefit", "work capacity"],
    "contract":  ["contract", "agreement", "breach", "liability", "damages", "obligation"],
    "criminal":  ["criminal", "penalty", "sentence", "offence", "conviction"],
    # --- NEW KEYWORDS ADDED HERE ---
    "inheritance": ["will", "testament", "inherit", "inheritance", "heir", 
                    "estate", "bequeath", "legacy", "testator"],
}

import re

def run_dag_pipeline(query, llm, embed_model, cross_encoder, lance_table, tools, config):
    """DAG-structured legal citation retrieval pipeline.

    Steps:
      1. Domain routing (Python only)
      2. Parallel retrieval: statute lookup + BM25 + HyDE dense
      3. RRF fusion
      4. Candidate citation collection
      5. Single LLM synthesis call
      6. Filter and return validated citations
    """
    # ── Step 1: Domain routing (Strict Keyword Counting) ──────────────────────
    q_lower = query.lower()
    
    domain_counts = {}
    for d, kws in DOMAIN_KEYWORDS.items():
        hits = 0
        for kw in kws:
            # \b ensures we match whole words, preventing "IV" from matching "surviving"
            if re.search(r'\b' + re.escape(kw.lower()) + r'\b', q_lower):
                hits += 1
        domain_counts[d] = hits
        
    best_domain = max(domain_counts, key=domain_counts.get)
    detected_domains = [best_domain] if domain_counts[best_domain] > 0 else []
    detected_domain = detected_domains[0] if detected_domains else None

    statute_keys: list[str] = []
    for d in detected_domains:
        for k in DOMAIN_STATUTE_MAP.get(d, []):
            if k not in statute_keys:
                statute_keys.append(k)

    # ── Step 2a: Direct statute lookup ───────────────────────────────────────
    statute_citations: list[str] = []
    lookup_tool = tools.get("lookup_citation")
    if lookup_tool:
        for key in statute_keys:
            try:
                obs = lookup_tool(key)
                if obs and "not found" not in obs.lower() and "error" not in obs.lower():
                    statute_citations.append(key)
            except Exception:
                pass

    # ── Step 2b & 2c & 3: Parallel Search & Correct RRF Fusion ───────────────
    concept_prompt = (
        "[INST] You are an expert in Swiss Law. Read the following case facts in English.\n"
        "Identify the core legal issues and translate them into formal Swiss legal terminology in German.\n"
        "For example, if the query mentions 'handwritten will' and 'avalanche', output 'Eigenhändige letztwillige Verfügung, Verfügungsfähigkeit, Erbrecht'.\n\n"
        f"Case Facts: {query}\n\n"
        "Output ONLY a comma-separated list of 3 to 5 formal German legal nouns or short phrases. Do not write full sentences. [/INST]"
    )

    sub_queries = []
    try:
        concept_resp = llm(
            concept_prompt,
            max_tokens=60,
            temperature=0.1,
            echo=False,
        )["choices"][0]["text"].strip()
        
        # Clean the response and split by commas
        cleaned_concepts = concept_resp.replace(".", "").replace("\n", ",")
        sub_queries = [c.strip() for c in cleaned_concepts.split(",") if c.strip()]
    except Exception:
        pass
        
    # Fallback just in case the LLM fails
    if not sub_queries:
        if config.get("use_query_decomposition"):
            sub_queries = decompose_query_to_subqueries(query, llm)
        else:
            sub_queries = [translate_query_to_keywords(query, llm)]

    # Dictionaries to accumulate proper RRF scores across ALL queries
    law_hit_dict = {}
    court_hit_dict = {}

    # 1. Process BM25 Searches using the new German Legal Concepts
    for subq in sub_queries:
        if not subq.strip():
            continue

        # Accumulate Law BM25 Scores
        for rank, doc in enumerate(laws_index.search(subq, top_k=config.get("top_k_laws", 15))):
            cit = doc.get("citation")
            if not cit: continue
            if cit not in law_hit_dict:
                law_hit_dict[cit] = {"doc": doc, "score": 0.0}
            law_hit_dict[cit]["score"] += 1.0 / (60 + rank + 1)

        # Accumulate Court BM25 Scores
        for rank, doc in enumerate(courts_index.search(subq, top_k=config.get("top_k_courts", 15))):
            cit = doc.get("citation")
            if not cit: continue
            if cit not in court_hit_dict:
                court_hit_dict[cit] = {"doc": doc, "score": 0.0}
            court_hit_dict[cit]["score"] += 1.0 / (60 + rank + 1)

    # ── Step 2c: Process HyDE Dense Search (Using the new concepts) ──────────
    try:
        hyde_instance = tools.get("hyde_search_courts")
        if hyde_instance is not None:
            # TIP APPLIED: Join the precise German legal concepts to build a stronger HyDE prompt
            full_german_query = ", ".join(sub_queries)
            hyp = hyde_instance._generate_hypothetical(full_german_query)
            
            if hyp:
                vec = embed_model.encode(
                    ["query: " + hyp],
                    normalize_embeddings=True,
                    show_progress_bar=False,
                )[0].tolist()
                
                df = (
                    lance_table.search(vec)
                    .nprobes(config.get("nprobes", 20))
                    .limit(config.get("top_k_hyde", 30))
                    .to_pandas()
                )
                if not df.empty:
                    for rank, doc in enumerate(df.to_dict("records")):
                        cit = doc.get("citation")
                        if not cit: continue
                        if cit not in court_hit_dict:
                            court_hit_dict[cit] = {"doc": doc, "score": 0.0}
                        # Merge dense scores into the court RRF pool
                        court_hit_dict[cit]["score"] += 1.0 / (60 + rank + 1)
    except Exception:
        pass

    # 3. Sort by final fused score
    fused_laws = [item["doc"] for item in sorted(law_hit_dict.values(), key=lambda x: x["score"], reverse=True)]
    fused_courts = [item["doc"] for item in sorted(court_hit_dict.values(), key=lambda x: x["score"], reverse=True)]


    # ── Step 3.5: Cross-Encoder Reranking ────────────────────────────────────
    if cross_encoder is not None and len(fused_courts) > 0:
        # Take the top 50 from RRF to rerank (preventing massive computation time)
        top_courts = fused_courts[:50]
        
        # Reconstruct a German query string from the sub-queries for the cross-encoder
        german_query = " ".join(sub_queries)
        
        # Prepare pairs: [query, document_text]
        # We truncate text to ~600 chars to fit within the cross-encoder's context limits
        pairs = [[german_query, doc.get("text", "")[:600]] for doc in top_courts]
        
        try:
            scores = cross_encoder.predict(pairs)
            for doc, score in zip(top_courts, scores):
                doc["cross_score"] = float(score)
                
            # Re-sort based purely on semantic relevance
            reranked_courts = sorted(top_courts, key=lambda x: x.get("cross_score", -100), reverse=True)
            
            # --- NEW: Dynamic Context Pruning ---
            # Standard cross-encoders output logits. Scores below a certain threshold (e.g., -2.0 or 0.0) 
            # signify no semantic entailment. Filter them out to keep the LLM context clean.
            filtered_courts = [doc for doc in reranked_courts if doc.get("cross_score", -100) > -1.5]
            
            # If the filter is too aggressive and removes everything, fall back to the top 5
            if not filtered_courts:
                filtered_courts = reranked_courts[:5]
                
            fused_courts = filtered_courts
        except Exception:
            pass

    # ── Step 3.7: Citation Graph Extraction (Free Recall) ────────────────────
    # This regex accurately captures Swiss law structures (e.g. Art. 221 Abs. 1 StPO)
    art_pattern_ext = r"Art\.\s+\d+[a-z]?\s+(?:(?:Abs\.|Ziff\.)\s+\d+\w*(?:\s+lit\.\s+\w+)?\s+)?[A-Z]\w+"
    
    # Scan the top 5 semantically reranked court decisions for laws they mention
    for doc in fused_courts[:5]:
        text = doc.get("text", "")
        extracted_laws = re.findall(art_pattern_ext, text)
        for law in extracted_laws:
            # Strip trailing whitespace/punctuation just in case
            clean_law = law.strip().rstrip(".,;")
            if clean_law not in statute_citations:
                statute_citations.append(clean_law)
                
    # ── Step 4: Collect all candidate citations ───────────────────────────────
    seen_cands: set[str] = set()
    all_candidates: list[str] = []
    for c in statute_citations:
        if c not in seen_cands:
            seen_cands.add(c); all_candidates.append(c)
    for doc in fused_laws + fused_courts:
        c = doc.get("citation", "")
        if c and c not in seen_cands:
            seen_cands.add(c); all_candidates.append(c)

    # ── Step 4.5: Critique Gate & Fallback Search ─────────────────────────────
    # Build a miniature context just for the critique (saves time and tokens)
    critique_ctx_parts = []
    for doc in fused_courts[:3]:
        critique_ctx_parts.append(doc.get("text", "")[:300])
    for doc in fused_laws[:2]:
        critique_ctx_parts.append(doc.get("text", "")[:300])
    critique_context = "\n\n".join(critique_ctx_parts)

    critique_prompt = (
        "[INST] You are a legal evaluation system. Read the user query and the retrieved context below.\n"
        f"Query: {query}\n\n"
        f"Context:\n{critique_context}\n\n"
        "Does the context contain relevant legal concepts, precedents, or statutes to answer the query? "
        "Answer ONLY 'YES' or 'NO'. [/INST]"
    )
    
    try:
        # Extremely fast call (max 10 tokens)
        critique_resp = llm(
            critique_prompt,
            max_tokens=10,
            temperature=0.0, # Deterministic check
            echo=False,
        )["choices"][0]["text"].strip().upper()
        
        # If the LLM rejects the context, trigger the fallback
        if "NO" in critique_resp:
            print("  [DAG] Context rejected by Critique. Executing broad fallback search...")
            
            # 1. Generate ultra-broad fallback keywords
            fallback_prompt = (
                "[INST] Extract exactly 1 or 2 core German nouns representing the absolute fundamental legal concept in this query. "
                "Output ONLY the nouns separated by a space.\n"
                f"Query: {query} [/INST]"
            )
            fallback_kws = llm(
                fallback_prompt,
                max_tokens=15,
                temperature=0.0,
                echo=False,
            )["choices"][0]["text"].strip()
            
            # 2. Brute-force BM25 search with the simplified keywords
            fb_laws = laws_index.search(fallback_kws, top_k=15)
            fb_courts = courts_index.search(fallback_kws, top_k=15)
            
            # 3. Dense fallback (if available)
            try:
                if tools.get("hyde_search_courts"):
                    vec = embed_model.encode(
                        ["query: " + fallback_kws], 
                        normalize_embeddings=True, 
                        show_progress_bar=False
                    )[0].tolist()
                    df = lance_table.search(vec).nprobes(20).limit(15).to_pandas()
                    if not df.empty:
                        fb_courts.extend(df.to_dict("records"))
            except Exception:
                pass
                
            # 4. Prepend the fallback results so they are seen FIRST in Step 5
            # Deduplicate while preserving order
            seen_fb = set()
            new_fused_laws = []
            for doc in fb_laws + fused_laws:
                cit = doc.get("citation")
                if cit and cit not in seen_fb:
                    seen_fb.add(cit)
                    new_fused_laws.append(doc)
            
            new_fused_courts = []
            for doc in fb_courts + fused_courts:
                cit = doc.get("citation")
                if cit and cit not in seen_fb:
                    seen_fb.add(cit)
                    new_fused_courts.append(doc)
                    
            fused_laws = new_fused_laws
            fused_courts = new_fused_courts
            
    except Exception:
        pass

    # ── Step 5: Synthesis LLM call ────────────────────────────────────────────
    # ── Step 5: Synthesis LLM call ────────────────────────────────────────────
    # ── Step 5: Hybrid Synthesis LLM call ─────────────────────────────────────
    # ── Step 5: Self-Consistency IRAC Synthesis (Majority Voting) ─────────────
    ctx_parts: list[str] = []
    
    # Prioritize Court decisions heavily, since LanceDB actually finds them
    for doc in fused_courts[:20]: 
        cit  = doc.get("citation", "")
        text = doc.get("text", "")[:400] 
        if cit:
            ctx_parts.append(f"[CITATION_KEY: {cit}]\n{text}")
            
    # Keep a few laws just in case BM25 gets lucky, but don't waste token space
    for doc in fused_laws[:5]:
        cit  = doc.get("citation", "")
        text = doc.get("text", "")[:300]
        if cit:
            ctx_parts.append(f"[CITATION_KEY: {cit}]\n{text}")
            
    context = "\n\n".join(ctx_parts)

    statute_str = "; ".join(statute_citations) if statute_citations else "none"
    
    # HYBRID PROMPT: Extract Courts from context, generate Statutes from memory, RAC structure
    synth_prompt = (
        "[INST] You are a Swiss legal expert. Analyze the case and list ALL relevant Swiss legal citations.\n\n"
        f"Query: {query}\n\n"
        f"Retrieved Court Decisions (Context):\n{context}\n\n"
        f"Domain statutes automatically identified: {statute_str}\n\n"
        "Task:\n"
        "1. ISSUE (Frage): State the primary legal question in German.\n"
        "2. RULE (Regel): Use your expert parametric knowledge of Swiss Law to list the governing federal statute articles (e.g., ZGB, OR, StPO, IVG).\n"
        "3. APPLICATION (Subsumtion): Explain exactly how the retrieved Court Decisions (Context) apply to the facts of the query.\n"
        "4. CONCLUSION (Schlussfolgerung): State the final legal outcome.\n"
        "5. CITATIONS: Output ONLY a single line starting with 'CITATIONS:' followed by a semicolon-separated list of the exact CITATION_KEYs used in steps 2 and 3.\n"
        "[/INST]"
    )

    
    
    ensemble_citations = []
    
    # Run the synthesis 3 times for majority voting
    for _ in range(3):
        try:
            synth_resp = llm(
                synth_prompt,
                max_tokens=600, 
                temperature=0.4, # INCREASED: allows the model to explore different reasoning paths
                stop=["</s>", "[INST]"], 
                echo=False,
            )["choices"][0]["text"].strip()
            
            # Parse the CoT output to find the exact CITATIONS line
            cit_line = ""
            for line in synth_resp.split("\n"):
                if line.strip().upper().startswith("CITATIONS:"):
                    cit_line = line.replace("CITATIONS:", "").replace("Citations:", "").strip()
                    break
            
            # Fallback if the model forgot the CITATIONS label
            if not cit_line:
                cit_line = synth_resp
                
            # Clean and split the extracted line
            cleaned_cit_line = cit_line.replace("-", ";").replace("*", ";")
            parsed_cits = [p.strip() for p in cleaned_cit_line.split(";") if p.strip()]
            run_cits = list(parsed_cits)

            # Regex Fallback (Scans the ENTIRE response, including the reasoning section)
            art_pattern = r"Art\.\s+\d+[a-z]?\s+(?:(?:Abs\.|Ziff\.)\s+\d+\w*(?:\s+lit\.\s+\w+)?\s+)?[A-Z]\w+"
            bge_pattern = r"BGE\s+\d{2,3}\s+[IVX]+\w*\s+\d+(?:\s+E\.\s+\d+(?:\.\d+)*)?"
            docket_pattern = r"\d+[A-Z]{1,2}_\d+/\d{4}(?:\s+E\.\s+\d+(?:\.\d+)*)?"

            prose_cits = (
                re.findall(art_pattern, synth_resp) +
                re.findall(bge_pattern, synth_resp) +
                re.findall(docket_pattern, synth_resp)
            )
            run_cits.extend(prose_cits)
            
            # CRITICAL: Deduplicate within this single run so a citation only gets 1 vote per run
            unique_run_cits = list(set(run_cits))
            ensemble_citations.extend(unique_run_cits)

        except Exception:
            pass

    # Tally the votes: only keep citations that were predicted in at least 2 out of 3 runs
    vote_counts = Counter(ensemble_citations)
    llm_citations = [cit for cit, count in vote_counts.items() if count >= 2]

    # ── Step 6: Filter and return ─────────────────────────────────────────────
    seen_final: set[str] = set()
    final_raw: list[str] = []
    for c in statute_citations + llm_citations:
        if c not in seen_final:
            seen_final.add(c); final_raw.append(c)

    validated = _filter_valid_citations(final_raw)

    print(f"  [DAG] domain={detected_domain or 'none':12s} | "
          f"sub_queries={len(sub_queries)} | citations={len(validated)}")

    return {"citations": validated, "sub_queries": sub_queries, "domain": detected_domain}


print("DAG pipeline defined. DOMAIN_STATUTE_MAP domains:", list(DOMAIN_STATUTE_MAP.keys()))


DAG pipeline defined. DOMAIN_STATUTE_MAP domains: ['detention', 'disability', 'contract', 'criminal', 'inheritance']


## 6. Load Test Data

In [118]:
import pandas as pd

# Load queries from the configured query file
if not QUERY_FILE.exists():
    raise FileNotFoundError(f"Query file not found: {QUERY_FILE}")

test_df = pd.read_csv(QUERY_FILE)

print(f"Loaded {len(test_df)} queries from {QUERY_FILE}")
print(f"Columns: {list(test_df.columns)}")

if IS_VALIDATION_MODE and "gold_citations" in test_df.columns:
    print(f"Gold citations available for evaluation")

test_df.head()

Loaded 10 queries from C:\Users\david\Desktop\project_ssl\Omnilex-Agentic-Retrieval-Competition\data\val.csv
Columns: ['query_id', 'query', 'gold_citations']
Gold citations available for evaluation


,query_id,query,gold_citations
0,val_001,May a court lawfully order a three‑month exten...,Art. 221 Abs. 1 StPO;Art. 140 Abs. 1 StGB;Art....
1,val_002,A claimant holding a national vocational diplo...,Art. 8 Abs. 1 ATSG;Art. 8 Abs. 1 IVG;Art. 17 A...
2,val_003,"A. Rivera, a Peruvian national born in 1994 an...",Art. 29 Abs. 2 BV;Art. 221 Abs. 1 StPO;Art. 39...
3,val_004,"Mr. Dalton, born in 1941 and resident in a sma...",Art. 505 Abs. 1 ZGB;Art. 467 ZGB;Art. 469 Abs....
4,val_005,"A parent, separated from their co-parent since...",Art. 133 Abs. 1 ZGB;Art. 133 Abs. 2 ZGB;Art. 2...


## 7. Generate Predictions

In [119]:
from tqdm import tqdm

predictions = []
all_logs    = []

QUERY_FILE_PATH = QUERY_FILE  # set in cell 2

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Running DAG pipeline"):
    qid   = row["query_id"]
    query = row["query"]

    result = run_dag_pipeline(
        query=query,
        llm=llm,
        embed_model=_embed_model,
        cross_encoder = _cross_encoder,
        lance_table=_lance_table,
        tools=TOOLS,
        config=CONFIG,
    )
    raw_citations = result["citations"]
    citations = _filter_valid_citations(raw_citations)

    predictions.append({
        "query_id":            qid,
        "predicted_citations": ";".join(citations),
    })
    all_logs.append({"query_id": qid, "query": query, "logs": result})

predictions_df = pd.DataFrame(predictions)
print(f"Done — {len(predictions_df)} predictions")
predictions_df.head()


Running DAG pipeline:   0%|          | 0/10 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

...

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Running DAG pipeline: 100%|██████████| 10/10 [37:09<00:00, 223.00s/it]

  [DAG] domain=criminal     | sub_queries=10 | citations=4
Done — 10 predictions


,query_id,predicted_citations
0,val_001,Art. 221 Abs. 1 StPO;Art. 221 Abs. 2 StPO;Art....
1,val_002,Art. 17 Abs. 1 IVG;Art. 8 Abs. 1 IVG;Art. 4 Ab...
2,val_003,Art. 221 Abs. 1 StPO;Art. 221 Abs. 2 StPO;Art....
3,val_004,Art. 467 ZGB;Art. 469 Abs. 1 ZGB;Art. 469 Abs....
4,val_005,Art. 221 Abs. 1 StPO;Art. 221 Abs. 2 StPO;Art....


## 8. Create Submission

In [120]:
# Save submission
submission_path = OUTPUT_PATH / "submission.csv"
predictions_df.to_csv(submission_path, index=False)

print(f"Submission saved to: {submission_path}")
print(f"Total predictions: {len(predictions_df)}")

# Show sample
print("\nSample submission:")
print(predictions_df.head())

Submission saved to: C:\Users\david\Desktop\project_ssl\Omnilex-Agentic-Retrieval-Competition\output\submission.csv
Total predictions: 10

Sample submission:
  query_id                                predicted_citations
0  val_001  Art. 221 Abs. 1 StPO;Art. 221 Abs. 2 StPO;Art....
1  val_002  Art. 17 Abs. 1 IVG;Art. 8 Abs. 1 IVG;Art. 4 Ab...
2  val_003  Art. 221 Abs. 1 StPO;Art. 221 Abs. 2 StPO;Art....
3  val_004  Art. 467 ZGB;Art. 469 Abs. 1 ZGB;Art. 469 Abs....
4  val_005  Art. 221 Abs. 1 StPO;Art. 221 Abs. 2 StPO;Art....


In [121]:
from collections.abc import Sequence


def citation_f1(
    predicted: Sequence[str],
    gold: Sequence[str],
) -> dict[str, float]:
    """Compute F1 score for citation overlap on a single query.

    Args:
        predicted: List of predicted canonical citation IDs
        gold: List of ground truth canonical citation IDs

    Returns:
        Dictionary with precision, recall, and F1
    """
    pred_set = set(predicted)
    gold_set = set(gold)

    # Edge case: both empty
    if len(pred_set) == 0 and len(gold_set) == 0:
        return {"precision": 1.0, "recall": 1.0, "f1": 1.0}

    # Edge case: prediction empty but gold not
    if len(pred_set) == 0:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}

    # Edge case: gold empty but prediction not
    if len(gold_set) == 0:
        return {"precision": 0.0, "recall": 1.0, "f1": 0.0}

    true_positives = len(pred_set & gold_set)
    precision = true_positives / len(pred_set)
    recall = true_positives / len(gold_set)

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    return {"precision": precision, "recall": recall, "f1": f1}


def macro_f1(
    predictions: Sequence[Sequence[str]],
    gold: Sequence[Sequence[str]],
) -> dict[str, float]:
    """Compute Macro F1: average F1 across all queries.

    This is the PRIMARY competition metric.

    Args:
        predictions: List of predicted citation lists (one per query)
        gold: List of gold citation lists (one per query)

    Returns:
        Dictionary with macro precision, recall, and F1
    """
    if len(predictions) != len(gold):
        raise ValueError(f"Length mismatch: {len(predictions)} predictions vs {len(gold)} gold")

    if len(predictions) == 0:
        return {"macro_precision": 0.0, "macro_recall": 0.0, "macro_f1": 0.0}

    precision_scores = []
    recall_scores = []
    f1_scores = []

    for pred, g in zip(predictions, gold):
        scores = citation_f1(pred, g)
        precision_scores.append(scores["precision"])
        recall_scores.append(scores["recall"])
        f1_scores.append(scores["f1"])

    n = len(f1_scores)
    return {
        "macro_precision": sum(precision_scores) / n,
        "macro_recall": sum(recall_scores) / n,
        "macro_f1": sum(f1_scores) / n,
    }


def micro_f1(
    predictions: Sequence[Sequence[str]],
    gold: Sequence[Sequence[str]],
) -> dict[str, float]:
    """Compute Micro F1: aggregate TP/FP/FN across all queries.

    Args:
        predictions: List of predicted citation lists (one per query)
        gold: List of gold citation lists (one per query)

    Returns:
        Dictionary with micro precision, recall, and F1
    """
    if len(predictions) != len(gold):
        raise ValueError(f"Length mismatch: {len(predictions)} predictions vs {len(gold)} gold")

    total_tp = 0
    total_fp = 0
    total_fn = 0

    for pred, g in zip(predictions, gold):
        pred_set = set(pred)
        gold_set = set(g)

        tp = len(pred_set & gold_set)
        fp = len(pred_set - gold_set)
        fn = len(gold_set - pred_set)

        total_tp += tp
        total_fp += fp
        total_fn += fn

    if total_tp + total_fp == 0:
        precision = 0.0
    else:
        precision = total_tp / (total_tp + total_fp)

    if total_tp + total_fn == 0:
        recall = 0.0
    else:
        recall = total_tp / (total_tp + total_fn)

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    return {
        "micro_precision": precision,
        "micro_recall": recall,
        "micro_f1": f1,
    }


def evaluate_submission(
    submission_df: pd.DataFrame,
    gold_df: pd.DataFrame,
    metrics: list[str] | None = None,
) -> dict[str, float]:
    """Evaluate a submission DataFrame against gold DataFrame.

    Args:
        submission_df: DataFrame with query_id and predicted_citations
        gold_df: DataFrame with query_id and gold_citations
        metrics: List of metrics to compute (default: all)

    Returns:
        Dictionary with requested metric scores
    """
    citation_separator = ";"
    
    def parse_citations(citation_string: str) -> list[str]:
        """Parse citation string into list (citations are already normalized)."""
        if not citation_string or citation_string.strip() == "":
            return []
        return [c.strip() for c in citation_string.split(citation_separator) if c.strip()]

    # Merge DataFrames
    merged = pd.merge(
        submission_df,
        gold_df,
        on="query_id",
        how="inner",
    )

    # Parse citations
    predictions = [
        parse_citations(row.get("predicted_citations", "")) for _, row in merged.iterrows()
    ]
    gold = [parse_citations(row.get("gold_citations", "")) for _, row in merged.iterrows()]

    # Compute all scores
    all_scores = {}

    macro_scores = macro_f1(predictions, gold)
    micro_scores = micro_f1(predictions, gold)

    all_scores.update(macro_scores)
    all_scores.update(micro_scores)

    # Log per-sample TP/FP/FN for each query
    print("\n" + "="*50)
    print("PER-SAMPLE EVALUATION RESULTS")
    print("="*50)
    for idx, (_, row) in enumerate(merged.iterrows()):
        query_id = row["query_id"]
        pred_set = set(predictions[idx])
        gold_set = set(gold[idx])
        
        true_positives = list(pred_set & gold_set)
        false_positives = list(pred_set - gold_set)
        false_negatives = list(gold_set - pred_set)
        
        print(f"\nQuery ID: {query_id}")
        print(f"  True Positives ({len(true_positives)}): {true_positives}")
        print(f"  False Positives ({len(false_positives)}): {false_positives}")
        print(f"  False Negatives ({len(false_negatives)}): {false_negatives}")
    
    print("\n" + "="*50)

    # Filter to requested metrics
    if metrics:
        metric_mapping = {
            "f1": "macro_f1",
            "precision": "macro_precision",
            "recall": "macro_recall",
            "macro_f1": "macro_f1",
            "micro_f1": "micro_f1",
        }
        filtered = {}
        for m in metrics:
            key = metric_mapping.get(m, m)
            if key in all_scores:
                filtered[m] = all_scores[key]
        return filtered

    return all_scores

## 9. Local Evaluation (Optional)

In [122]:
# Evaluate if in validation mode with gold labels
if IS_VALIDATION_MODE and "gold_citations" in test_df.columns:
    # Join predictions with gold citations from the same file
    eval_df = predictions_df.merge(
        test_df[["query_id", "gold_citations"]],
        on="query_id",
        how="inner"
    )
    
    if len(eval_df) > 0:
        scores = evaluate_submission(
            eval_df[["query_id", "predicted_citations"]],
            eval_df[["query_id", "gold_citations"]],
        )
        
        print("\n" + "="*50)
        print("EVALUATION RESULTS")
        print("="*50)
        print(f"Queries evaluated: {len(eval_df)}")
        print(f"\nMacro F1 (PRIMARY): {scores['macro_f1']:.4f}")
        print(f"Macro Precision:    {scores['macro_precision']:.4f}")
        print(f"Macro Recall:       {scores['macro_recall']:.4f}")
        print(f"\nMicro F1:           {scores['micro_f1']:.4f}")
        print(f"Micro Precision:    {scores['micro_precision']:.4f}")
        print(f"Micro Recall:       {scores['micro_recall']:.4f}")
    else:
        print("No overlapping queries for evaluation.")
else:
    print("Skipping evaluation (not in validation mode or no gold labels available)")


PER-SAMPLE EVALUATION RESULTS

Query ID: val_001
  True Positives (5): ['Art. 221 Abs. 2 StPO', 'Art. 227 Abs. 1 StPO', 'Art. 221 Abs. 1 StPO', 'Art. 212 Abs. 3 StPO', 'Art. 100 Abs. 1 BGG']
  False Positives (8): ['BGE 146 IV 136 E. 2.10', 'Art. 221 Abs. 1 lit. b StPO', 'Art. 228 Abs. 1 StPO', 'Art. 226 Abs. 1 StPO', 'Art. 237 Abs. 1 StPO', 'Art. 221 Abs. 1 lit. a StPO', 'Art. 231 Abs. 1 StPO', 'Art. 221 Abs. 1 lit. c StPO']
  False Negatives (37): ['Art. 37 Abs. 1 StBOG', '7B_301/2024 E. 2.4', 'Art. 382 Abs. 1 StPO', 'Art. 422 Abs. 1 StPO', 'BGE 137 IV 122 E. 4.1', 'BGE 143 IV 168 E. 5.1', 'BGE 139 IV 270 E. 3.1', 'BGE 137 IV 122 E. 6.2', 'BGE 137 IV 122 E. 4.2', '1B_357/2022 E. 3.1', '7B_69/2024 E. 3.3.2', '7B_496/2025 E. 3.2', '1B_210/2023 E. 4.1', 'BGE 132 I 21 E. 3.2.1', 'Art. 385 Abs. 1 StPO', 'Art. 393 Abs. 1 StPO', '1B_536/2018 E. 5.1', 'Art. 222 StPO', '7B_231/2025 E. 4.1', '1B_15/2023 E. 3.1', 'Art. 422 Abs. 2 StPO', 'Art. 135 Abs. 4 StPO', 'BGE 133 I 168 E. 4.1', 'Art. 39 

## 10. Failure Diagnosis

Runs 3 sample queries and diagnoses which failure mode dominates: retrieval gap, extraction failure, context bloat, or timeout.

In [80]:
import time
import textwrap

# ── Diagnostic configuration ──────────────────────────────────────────
DIAG_N       = 3    # number of val queries to diagnose
ORACLE_K     = 10   # BM25 top-K for oracle recall
TIMEOUT_SEC  = 180  # flag as timeout if agent exceeds this


def _tokens(text: str) -> int:
    return max(1, len(text) // 4)


def _oracle_recall(query: str, gold: list[str], k: int) -> dict:
    """What fraction of gold citations appear in BM25 top-K?"""
    law_hits   = {r['citation'] for r in laws_index.search(query, top_k=k)}
    court_hits = {r['citation'] for r in courts_index.search(query, top_k=k)}
    all_hits   = law_hits | court_hits
    gold_set   = set(gold)
    found      = gold_set & all_hits
    return {
        'recall':      len(found) / len(gold_set) if gold_set else 0.0,
        'found':       sorted(found),
        'missed':      sorted(gold_set - all_hits),
        'law_found':   sorted(gold_set & law_hits),
        'court_found': sorted(gold_set & court_hits),
    }


def _run_timed(query: str):
    """Run agent; return (citations, logs, elapsed_sec, flagged_timeout)."""
    t0 = time.time()
    try:
        cits, logs = run_agent(query, verbose=False)
        elapsed    = time.time() - t0
        return cits, logs, elapsed, elapsed > TIMEOUT_SEC
    except Exception:
        return [], [], time.time() - t0, True


# ── Main diagnostic loop ───────────────────────────────────────────────
if not IS_VALIDATION_MODE or 'gold_citations' not in test_df.columns:
    print('Diagnosis requires validation mode with gold_citations.')
else:
    sample_df = test_df.head(DIAG_N).reset_index(drop=True)
    lines     = []
    SEP_H     = '=' * 72
    SEP_L     = '-' * 72

    lines += [
        SEP_H,
        '  FAILURE DIAGNOSIS REPORT',
        f"  Generated : {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}",
        f"  Queries   : {DIAG_N}   Oracle-K : {ORACLE_K}   Timeout : {TIMEOUT_SEC}s",
        SEP_H,
    ]

    summary_rows = []

    for _, row in sample_df.iterrows():
        qid      = row['query_id']
        query    = row['query']
        gold_str = row.get('gold_citations', '')
        gold     = [c.strip() for c in gold_str.split(';') if c.strip()]

        lines += ['', SEP_L, f'  QUERY  {qid}', SEP_L]
        lines.append(textwrap.fill(query, width=70, initial_indent='  '))

        # Gold citations
        lines += [f'\n  GOLD CITATIONS  ({len(gold)} total)']
        for g in gold:
            lines.append(f'    . {g}')

        # BM25 oracle recall
        oracle = _oracle_recall(query, gold, k=ORACLE_K)
        lines += [
            f"\n  BM25 ORACLE RECALL@{ORACLE_K}  ->  "
            f"{oracle['recall']:.0%}  ({len(oracle['found'])}/{len(gold)} gold retrievable)",
        ]
        if oracle['law_found']:
            lines.append(f"    Law hits   : {oracle['law_found']}")
        if oracle['court_found']:
            lines.append(f"    Court hits : {oracle['court_found']}")
        if oracle['missed']:
            lines.append(f"    MISSED     : {oracle['missed']}")

        # Run agent
        lines.append('\n  RUNNING AGENT ...')
        cits, logs, elapsed, flagged = _run_timed(query)

        status = f'TIMEOUT (>{TIMEOUT_SEC}s)' if flagged else 'COMPLETED'
        lines.append(f'  STATUS  {status}  ({elapsed:.1f}s)')

        # Raw citations extracted
        gold_set = set(gold)
        lines += [f'\n  RAW CITATIONS EXTRACTED  ({len(cits)})']
        for c in cits:
            tag = 'CORRECT' if c in gold_set else 'wrong'
            lines.append(f'    [{tag}] {c}')
        if not cits:
            lines.append('    (none)')

        # Observation token breakdown
        obs_logs         = [lg for lg in logs if lg.get('type') == 'tool_execution']
        total_obs_tokens = sum(_tokens(lg.get('observation', '')) for lg in obs_logs)
        lines += [f'\n  OBSERVATION TOKENS  (est. total: {total_obs_tokens})']
        for lg in obs_logs:
            toks = _tokens(lg.get('observation', ''))
            q50  = lg['query'][:50]
            lines.append(f"    {lg['tool']:15s}  query='{q50}'  -> {toks} tok")

        # Failure mode flags
        correct    = [c for c in cits if c in gold_set]
        f_retrieve = oracle['recall'] == 0.0
        f_partial  = 0.0 < oracle['recall'] < 0.5
        f_extract  = oracle['recall'] > 0 and len(correct) == 0
        f_context  = total_obs_tokens > 2000
        f_timeout  = flagged

        lines += ['\n  FAILURE MODE FLAGS']
        lines.append(f"    Retrieval failure  (0% oracle)           : {'YES <<' if f_retrieve else 'no'}")
        lines.append(f"    Partial retrieval  (<50% oracle)         : {'YES <<' if f_partial  else 'no'}")
        lines.append(f"    Extraction failure (BM25 found, missed)  : {'YES <<' if f_extract  else 'no'}")
        lines.append(f"    Context bloat      (obs tokens >2000)    : {'YES <<' if f_context  else 'no'} ({total_obs_tokens} tok)")
        lines.append(f"    Timeout                                  : {'YES <<' if f_timeout  else 'no'}")

        dominant = (
            'RETRIEVAL'  if f_retrieve else
            'EXTRACTION' if f_extract  else
            'TIMEOUT'    if f_timeout  else
            'CONTEXT'    if f_context  else
            'OK'
        )
        summary_rows.append({
            'id':         qid,
            'oracle':     f"{oracle['recall']:.0%}",
            'extracted':  len(cits),
            'correct':    len(correct),
            'obs_tokens': total_obs_tokens,
            'elapsed':    f'{elapsed:.0f}s',
            'dominant':   dominant,
        })

    # Summary table
    lines += ['', SEP_H, '  SUMMARY', SEP_H]
    lines.append(f"  {'ID':<12} {'Oracle':>8} {'Extracted':>10} {'Correct':>8} {'ObsTok':>8} {'Time':>7}  Dominant")
    lines.append('  ' + '-' * 66)
    for r in summary_rows:
        lines.append(
            f"  {r['id']:<12} {r['oracle']:>8} {r['extracted']:>10} "
            f"{r['correct']:>8} {r['obs_tokens']:>8} {r['elapsed']:>7}  {r['dominant']}"
        )

    dominant_counts = {}
    for r in summary_rows:
        dominant_counts[r['dominant']] = dominant_counts.get(r['dominant'], 0) + 1
    lines += ['', '  Dominant failure mode across all queries:']
    for mode, cnt in sorted(dominant_counts.items(), key=lambda x: -x[1]):
        lines.append(f'    {mode}: {cnt}/{DIAG_N}')
    lines += ['', SEP_H, '']

    report = '\n'.join(lines)
    print(report)

    out = OUTPUT_PATH / 'failure_diagnosis.txt'
    out.write_text(report, encoding='utf-8')
    print(f'Report saved -> {out}')


BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

  FAILURE DIAGNOSIS REPORT
  Generated : 2026-04-27 10:01
  Queries   : 3   Oracle-K : 10   Timeout : 180s

------------------------------------------------------------------------
  QUERY  val_001
------------------------------------------------------------------------
  May a court lawfully order a three‑month extension of pre‑trial
detention under Art. 221 Abs. 1 lit. b StPO (risk of collusion)
consistent with the principle of proportionality when the
accused—detained after an alleged late‑night assault and theft of a
courier satchel containing, inter alia, €5,600—was remanded by an
order dated 18 October 2024 for a maximum period up to 15 January
2025, the prosecutor sought an extension on 10 December 2024 primarily
citing a concrete risk that the detainee would influence witnesses or
tamper with evidence and a risk of reoffending, while the detainee
opposed the extension on the ground that most witnesses have already
been interviewed, the investigative steps still pending are
esse

## Summary

This agentic retrieval baseline demonstrates a more sophisticated approach:

1. **Tool-augmented generation**: The LLM can search actual legal corpora rather than relying solely on parametric knowledge.

2. **ReAct-style reasoning**: The agent reasons about what to search, executes searches, observes results, and iterates.

3. **Grounded citations**: Citations are extracted from actual search results, reducing hallucination.

4. **Comprehensive search**: The agent searches both laws and court decisions for complete results.

## Potential Improvements

- **Better search**: Use semantic search (embeddings) instead of BM25
- **Query expansion**: Generate multiple search queries in different languages
- **Relevance filtering**: Add a step to verify citations are actually relevant
- **Citation validation**: Check that generated citations exist in the corpus
- **Multi-hop reasoning**: Follow citation chains to find related sources

In [123]:
# Load test set
TEST_QUERY_FILE = DATA_PATH / "test.csv"

if TEST_QUERY_FILE.exists():
    print(f"Loading test set from {TEST_QUERY_FILE}")
    test_set_df = pd.read_csv(TEST_QUERY_FILE)
    print(f"Loaded {len(test_set_df)} test queries")
    print(f"Columns: {list(test_set_df.columns)}")
    
    # Generate predictions for test set
    test_predictions = []
    test_all_logs = []  # Store logs for all test queries
    
    print("\n" + "="*50)
    print("RUNNING DAG PIPELINE ON TEST SET")
    print("="*50)
    
    for _, row in tqdm(test_set_df.iterrows(), total=len(test_set_df), desc="Running DAG pipeline on test set"):
        query_id = row["query_id"]
        query_text = row["query"]
        
        # Run new DAG pipeline
        result = run_dag_pipeline(
            query=query_text,
            llm=llm,
            embed_model=_embed_model,
            cross_encoder = _cross_encoder,
            lance_table=_lance_table,
            tools=TOOLS,
            config=CONFIG,
        )
        
        raw_citations = result["citations"]
        citations = _filter_valid_citations(raw_citations)
        
        # Store logs with query_id
        test_all_logs.append({
            "query_id": query_id,
            "query": query_text,
            "logs": result,
        })
        
        test_predictions.append({
            "query_id": query_id,
            "predicted_citations": ";".join(citations),
        })
    
    print(f"\nGenerated predictions for {len(test_predictions)} test queries")
    print(f"Collected logs for {len(test_all_logs)} test queries")
    
    # Create DataFrame and save test submission
    test_predictions_df = pd.DataFrame(test_predictions)
    test_submission_path = OUTPUT_PATH / "test_submission.csv"
    test_predictions_df.to_csv(test_submission_path, index=False)
    
    print(f"\nTest submission saved to: {test_submission_path}")
    print(f"Total test predictions: {len(test_predictions_df)}")
    print("\nSample test submission:")
    print(test_predictions_df.head())
else:
    print(f"Test set file not found: {TEST_QUERY_FILE}")
    print("Skipping test set processing.")

Loading test set from C:\Users\david\Desktop\project_ssl\Omnilex-Agentic-Retrieval-Competition\data\test.csv
Loaded 40 test queries
Columns: ['query_id', 'query']

RUNNING DAG PIPELINE ON TEST SET


Running DAG pipeline on test set:   0%|          | 0/40 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

...

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Running DAG pipeline on test set: 100%|██████████| 40/40 [2:24:31<00:00, 216.78s/it]

  [DAG] domain=contract     | sub_queries=10 | citations=11

Generated predictions for 40 test queries
Collected logs for 40 test queries

Test submission saved to: C:\Users\david\Desktop\project_ssl\Omnilex-Agentic-Retrieval-Competition\output\test_submission.csv
Total test predictions: 40

Sample test submission:
   query_id                                predicted_citations
0  test_001  Art. 6 Abs. 5 ZPO;Art. 25 DSG;Art. 25 Abs. 3 l...
1  test_002  Art. 17 Abs. 1 IVG;Art. 8 Abs. 1 IVG;Art. 4 Ab...
2  test_003  Art. 1 Abs. 1 OR;Art. 18 Abs. 1 OR;Art. 97 Abs...
3  test_004  Art. 166 Abs. 2 SchKG;Art. 158 Abs. 2 SchKG;Ar...
4  test_005  Art. 467 ZGB;Art. 469 Abs. 1 ZGB;Art. 469 Abs....
